# Assembler Node — Unit Tests

The Assembler is deterministic (no LLM, no RAG) so the tests are pure:
  1. deep-merges the fragment into `final_openapi`,
  2. advances the loop (idx +1, counters reset, scratch fields cleared),
  3. tracks the fragment in `validated_fragments_by_op`,
  4. on the LAST operation, writes the YAML to `openapi_target_path`.

All inputs are hand-built in the notebook — no real spec needed.

In [1]:
# Step 1 — Imports and a tmp directory for YAML output

import tempfile
from pathlib import Path

import yaml

from openapi_generator.config import get_logger
from openapi_generator.graph.conditions import MAX_OP_ITERATIONS
from openapi_generator.nodes.assembler import assembler_node

logger = get_logger(__name__)

TMP = Path(tempfile.mkdtemp(prefix="assembler_test_"))
OUT = TMP / "out.yaml"
logger.info(f"Tmp output: {OUT}")

/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_generator/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-25 22:52:19 [INFO] __main__: Tmp output: /tmp/assembler_test_afey13r0/out.yaml


In [2]:
# Step 2 — Helpers to build state and fragments

def initial_state(plan, idx=0, fragment=None, errors=None, iteration=0,
                  seed_paths=None, seed_schemas=None, target=str(OUT)):
    """Mimic what would reach the Assembler after Loader + ... + Validator."""
    return {
        "operations_plan": plan,
        "current_op_idx": idx,
        "op_iteration_count": iteration,
        "reflected_fragment": fragment or {},
        "validation_errors": errors or [],
        "final_openapi": {
            "openapi": "3.0.3",
            "info": {"title": "Test"},
            "paths": dict(seed_paths or {}),
            "components": {"schemas": dict(seed_schemas or {})},
        },
        "openapi_target_path": target,
        "validated_fragments_by_op": {},
    }

def make_fragment(path, method, op_def, schemas=None):
    """Shape returned by the Patcher → Reflector → Validator chain."""
    return {
        "path": path,
        "method": method,
        "paths": {path: {method: op_def}},
        "components": {"schemas": dict(schemas or {})},
    }

TWO_OPS = [
    {"path": "/items", "method": "get", "action": "create", "source_rule_ids": [1, 2], "priority": "high", "rationale": ""},
    {"path": "/items/{id}", "method": "get", "action": "create", "source_rule_ids": [3], "priority": "medium", "rationale": ""},
]
logger.info(f"Sample plan: {TWO_OPS}")

2026-05-25 22:52:19 [INFO] __main__: Sample plan: [{'path': '/items', 'method': 'get', 'action': 'create', 'source_rule_ids': [1, 2], 'priority': 'high', 'rationale': ''}, {'path': '/items/{id}', 'method': 'get', 'action': 'create', 'source_rule_ids': [3], 'priority': 'medium', 'rationale': ''}]


## Scenario A — merge two ops sequentially, write YAML on the last

Two operations, each with its own fragment. After op 1 the loop advances; after op 2 the YAML is written.

In [3]:
# Step 3 — First call (idx=0): merge /items GET, advance to idx=1, no YAML yet

frag1 = make_fragment(
    "/items", "get",
    {"summary": "List items", "responses": {"200": {"description": "OK"}}},
    schemas={"Item": {"type": "object", "properties": {"id": {"type": "string"}}}},
)
state1 = initial_state(TWO_OPS, idx=0, fragment=frag1)
out1 = assembler_node(state1)

assert out1["current_op_idx"] == 1
assert out1["op_iteration_count"] == 0
assert out1["current_fragment"] == {}
assert out1["reflected_fragment"] == {}
assert out1["validation_errors"] == []
assert out1["final_output_path"] == "", "YAML must NOT be written before the last op"

merged_paths = out1["final_openapi"]["paths"]
merged_schemas = out1["final_openapi"]["components"]["schemas"]
assert "/items" in merged_paths and "get" in merged_paths["/items"]
assert "Item" in merged_schemas

logger.info(f"After op 1: paths={list(merged_paths)}, schemas={list(merged_schemas)}")
logger.info(f"validated_fragments_by_op: {list(out1['validated_fragments_by_op'])}")

2026-05-25 22:52:19 [INFO] openapi_generator.nodes.assembler: Assembler → merged get /items: +1 path block(s), +1 schema(s)
2026-05-25 22:52:19 [INFO] __main__: After op 1: paths=['/items'], schemas=['Item']
2026-05-25 22:52:19 [INFO] __main__: validated_fragments_by_op: ['get /items']


In [4]:
# Step 4 — Second call (idx=1, last op): merge /items/{id} GET, write YAML

frag2 = make_fragment(
    "/items/{id}", "get",
    {"summary": "Get item", "responses": {"200": {"description": "OK"}}},
    schemas={"ItemDetail": {"type": "object"}},
)
# Feed in the accumulated final_openapi from the previous call:
state2 = {
    **initial_state(TWO_OPS, idx=1, fragment=frag2),
    "final_openapi": out1["final_openapi"],
    "validated_fragments_by_op": out1["validated_fragments_by_op"],
}
out2 = assembler_node(state2)

assert out2["current_op_idx"] == 2  # past the end
assert out2["final_output_path"], "YAML path must be reported on the last op"

paths = out2["final_openapi"]["paths"]
schemas = out2["final_openapi"]["components"]["schemas"]
assert {"/items", "/items/{id}"} <= set(paths)
assert {"Item", "ItemDetail"} <= set(schemas)

# File must exist on disk and round-trip via YAML
written = Path(out2["final_output_path"])
assert written.is_file()
reloaded = yaml.safe_load(written.read_text(encoding="utf-8"))
assert reloaded["openapi"] == "3.0.3"
assert {"/items", "/items/{id}"} <= set(reloaded["paths"])
logger.info(f"Final YAML written and reloaded OK: {written}")

2026-05-25 22:52:19 [INFO] openapi_generator.nodes.assembler: Assembler → merged get /items/{id}: +1 path block(s), +1 schema(s)
2026-05-25 22:52:19 [INFO] openapi_generator.nodes.assembler: Assembler → wrote final OpenAPI to /tmp/assembler_test_afey13r0/out.yaml
2026-05-25 22:52:19 [INFO] __main__: Final YAML written and reloaded OK: /tmp/assembler_test_afey13r0/out.yaml


## Scenario B — schema name collision keeps the existing one

If the Patcher emits a schema with a name that already exists under `components.schemas` (e.g. from the legacy seed), the Assembler keeps the existing definition and logs a warning.

In [5]:
# Step 5 — Force a collision on schema name 'Item'

EXISTING_ITEM = {"type": "object", "properties": {"legacy_field": {"type": "string"}}}
frag_clash = make_fragment(
    "/items", "post",
    {"summary": "Create item", "responses": {"201": {"description": "Created"}}},
    schemas={"Item": {"type": "object", "properties": {"new_field": {"type": "integer"}}}},
)
state_clash = initial_state(
    [{"path": "/items", "method": "post", "action": "create", "source_rule_ids": [], "priority": "medium", "rationale": ""}],
    idx=0,
    fragment=frag_clash,
    seed_schemas={"Item": EXISTING_ITEM},
    target=str(TMP / "clash.yaml"),
)
out_clash = assembler_node(state_clash)

kept = out_clash["final_openapi"]["components"]["schemas"]["Item"]
assert kept == EXISTING_ITEM, "Existing schema must be preserved on collision"
logger.info("Schema collision: existing definition preserved. OK.")

2026-05-25 22:52:19 [WARNING] openapi_generator.nodes.assembler: Schema conflict at components.schemas.Item: keeping existing definition, discarding new one from this fragment.
2026-05-25 22:52:19 [INFO] openapi_generator.nodes.assembler: Assembler → merged post /items: +1 path block(s), +1 schema(s)
2026-05-25 22:52:19 [INFO] openapi_generator.nodes.assembler: Assembler → wrote final OpenAPI to /tmp/assembler_test_afey13r0/clash.yaml
2026-05-25 22:52:19 [INFO] __main__: Schema collision: existing definition preserved. OK.


## Scenario C — force-include at MAX_OP_ITERATIONS

If the retry budget is exhausted but corrections remain, the fragment is accepted anyway and tagged with `x-openapi-gen-warnings` on every operation block.

In [6]:
# Step 6 — iteration = MAX_OP_ITERATIONS with leftover correction errors

frag_warn = make_fragment(
    "/items", "delete",
    {"summary": "Delete", "responses": {"204": {"description": "No content"}}},
)
state_warn = initial_state(
    [{"path": "/items", "method": "delete", "action": "create", "source_rule_ids": [], "priority": "low", "rationale": ""}],
    idx=0,
    fragment=frag_warn,
    iteration=MAX_OP_ITERATIONS,
    errors=[
        {"error_type": "correction", "stage": "semantic_2",
         "instruction": "Missing 404 response", "ref": "$.paths./items.delete.responses"},
    ],
    target=str(TMP / "warn.yaml"),
)
out_warn = assembler_node(state_warn)

delete_op = out_warn["final_openapi"]["paths"]["/items"]["delete"]
assert "x-openapi-gen-warnings" in delete_op
assert any("Missing 404" in w for w in delete_op["x-openapi-gen-warnings"])
logger.info(f"Force-include warnings: {delete_op['x-openapi-gen-warnings']}")

2026-05-25 22:52:19 [WARNING] openapi_generator.nodes.assembler: Assembler → force-include delete /items at max retries with 1 unresolved correction(s)
2026-05-25 22:52:19 [INFO] openapi_generator.nodes.assembler: Assembler → merged delete /items: +1 path block(s), +0 schema(s)
2026-05-25 22:52:19 [INFO] openapi_generator.nodes.assembler: Assembler → wrote final OpenAPI to /tmp/assembler_test_afey13r0/warn.yaml
2026-05-25 22:52:19 [INFO] __main__: Force-include warnings: ['Missing 404 response']


## Scenario D — action='discard' removes the legacy op

When the Planner marks a legacy operation as `discard`, the Assembler pops
it from `final_openapi.paths[path][method]` (and drops the empty path item
if no methods remain). The Patcher is bypassed, so the fragment is empty.

In [ ]:
# Step 6b — discard: legacy op is removed from final_openapi

DISCARD_PLAN = [
    {"path": "/items", "method": "delete", "action": "discard",
     "source_rule_ids": [], "priority": "low", "rationale": "deprecated in Rel-18"},
]
state_discard = initial_state(
    DISCARD_PLAN,
    idx=0,
    fragment={},
    seed_paths={
        "/items": {
            "get": {"summary": "List", "responses": {"200": {"description": "OK"}}},
            "delete": {"summary": "Drop", "responses": {"204": {"description": "No content"}}},
        }
    },
    target=str(TMP / "discard.yaml"),
)
out_discard = assembler_node(state_discard)
items = out_discard["final_openapi"]["paths"]["/items"]
assert "delete" not in items, "delete must be removed on discard"
assert "get" in items, "other methods on the same path must be preserved"
logger.info(f"After discard: /items methods left = {list(items)}")

## Failure-mode / DI tests

In [7]:
# Step 7 — Empty fragment still advances the loop (e.g. Patcher stub)

state_empty = initial_state(
    [{"path": "/x", "method": "get", "action": "create", "source_rule_ids": [], "priority": "low", "rationale": ""}],
    idx=0,
    fragment={},
    target=str(TMP / "empty.yaml"),
)
out_empty = assembler_node(state_empty)
assert out_empty["current_op_idx"] == 1
assert out_empty["final_openapi"]["paths"] == {}
# Single-op plan → idx 1 is past the end → YAML written
assert out_empty["final_output_path"]
logger.info("Empty fragment: loop advanced, YAML written. OK.")

2026-05-25 22:52:19 [INFO] openapi_generator.nodes.assembler: Assembler → merged get /x: +0 path block(s), +0 schema(s)
2026-05-25 22:52:19 [INFO] openapi_generator.nodes.assembler: Assembler → wrote final OpenAPI to /tmp/assembler_test_afey13r0/empty.yaml
2026-05-25 22:52:19 [INFO] __main__: Empty fragment: loop advanced, YAML written. OK.


In [8]:
# Step 8 — DI uniformity: accepts llm/retriever kwargs and ignores them

out_di = assembler_node(
    initial_state([], idx=0, fragment={}, target=str(TMP / "di.yaml")),
    llm=object(),
    retriever=object(),
)
assert out_di["current_op_idx"] == 1
logger.info("Assembler accepts llm/retriever kwargs without using them. OK.")

2026-05-25 22:52:19 [INFO] openapi_generator.nodes.assembler: Assembler → merged ? ?: +0 path block(s), +0 schema(s)
2026-05-25 22:52:19 [INFO] openapi_generator.nodes.assembler: Assembler → wrote final OpenAPI to /tmp/assembler_test_afey13r0/di.yaml
2026-05-25 22:52:19 [INFO] __main__: Assembler accepts llm/retriever kwargs without using them. OK.
